In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/resume.csv")

df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
df.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')

In [4]:
df.shape

(2484, 4)

In [5]:
sample_cv = df.loc[0, "Resume_str"]

print(sample_cv)

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

In [8]:
import os 
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENROUTER_API_KEY")

print("API key tersedia", api_key is not None)

API key tersedia True


In [94]:
import requests
import json

def extract_cv_skills(cv_text):

    prompt = f"""
You are an expert technical recruiter and resume analyst.

Extract the skills and competencies explicitly demonstrated or stated
in the resume below.

Include:
- technical skills
- software and tools
- technologies
- professional competencies
- domain-specific knowledge
- methodologies
- relevant business skills

Rules:

1. Extract only skills that are explicitly stated in the resume.
2. Do not infer skills that are not mentioned.
3. Prefer concise skill names.
4. Normalize similar concepts into one standard skill.
5. Do NOT include:
   - job titles
   - company names
   - years of experience
   - dates
   - educational degrees
   - personality traits
   - complete sentences
6. Combine duplicate or highly similar skills.
7. Keep skill names concise, normally 1–4 words.
8. Return approximately 10–30 of the most relevant skills.
9. Return ONLY a valid JSON array of strings.
10. Do not include markdown, explanations, or additional text.

Resume:
{cv_text}
"""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "nvidia/nemotron-3.5-lightning:free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0
        }
    )

    if response.status_code != 200:
        print("Status code:", response.status_code)
        print("Response:", response.text)
        return None

    result = response.json()

    if "choices" not in result:
        print("Response tidak memiliki 'choices':")
        print(result)
        return None

    content = result["choices"][0]["message"].get("content")

    if content is None:
        print("Model tidak mengembalikan content.")
        print(result)
        return []

    content = content.strip()

    if content.startswith("```"):
        content = content.replace("```json", "")
        content = content.replace("```", "")
        content = content.strip()

    return json.loads(content)

In [11]:
cv_skills = extract_cv_skills(sample_cv)

cv_skills

['Customer Service',
 'Team Management',
 'Marketing & Advertising',
 'Conflict Resolution',
 'Training & Development',
 'Client Relations',
 'Hospitality Property Management Systems',
 'Medical Billing & Coding',
 'Medical Terminology',
 'Data Analysis',
 'Benefits Administration',
 'Employee Relations',
 'Labor Relations',
 'Policy Development',
 'Payroll Administration',
 'Financial Management',
 'Budgeting',
 'Accounting',
 'Purchasing & Procurement',
 'Public Relations',
 'Government Relations',
 'Graphic Design',
 'Website Development',
 'Social Media Management',
 'Competitor Analysis',
 'Inventory Control',
 'Loss Prevention',
 'Safety Compliance',
 'Time Management',
 'Leadership']

In [25]:
roles_df = pd.read_excel(
    "../data/company_roles/company_roles_with_skills.xlsx"
)

roles_df[
    ["role", "department", "extracted_skills"]
]

,role,department,extracted_skills
0,AI and Data Specialist,Information Technology,"['Full-stack engineering', 'Artificial Intelli..."
1,Junior Payroll Specialist,Finance,"['Payroll Processing', 'Payroll Compliance', '..."
2,Technical Support,Finance,"['Information Systems', 'Logistics Management'..."
3,SEO Associate,Marketing,"['SEO Campaign Optimization', 'SEO Writing', '..."
4,Solution Engineering,Information Technology,"['JavaScript', 'CSS', 'REST APIs', 'Database M..."
5,Risk Management Specialist,Legal,"['Risk Management', 'SaaS Domain Knowledge', '..."
6,Project Manager,Marketing,"['Project Management', 'ERP Implementation', '..."
7,IT Auditor Lead,Information Technology,"['IT Audit', 'IT Risk Management', 'Cybersecur..."
8,Senior QA Engineer,Information Technology,"['Selenium', 'Playwright', 'Cypress', 'Appium'..."


In [26]:
print(roles_df["extracted_skills"].iloc[0])
print(type(roles_df["extracted_skills"].iloc[0]))

['Full-stack engineering', 'Artificial Intelligence (AI)', 'Machine Learning (ML)', 'Large Language Models (LLM)', 'Retrieval-Augmented Generation (RAG)', 'Prompt engineering', 'Retrieval systems', 'Orchestration', 'Data system design', 'Data pipelines', 'Data quality', 'Data lineage', 'Evaluation design', 'Golden datasets', 'Regression evaluation', 'Scenario testing', 'Grounding', 'Hallucination control', 'Guardrails', 'AI safety', 'Model monitoring', 'Drift detection', 'Production monitoring', 'Agentic workflows', 'Agent optimization', 'Non-deterministic systems', 'AI quality management', 'Production AI systems']
<class 'str'>


- hasil extraction skill berupa list,
- ketika dimasukkan ke excel dengan dataframe, tetap list
- namun ketika file excelnya dipanggil, tipe datanya berubah jadi string
- makanya harus di ubah menjadi list lagi dengan menggunakan ast

In [27]:
import ast

roles_df["extracted_skills"] = roles_df["extracted_skills"].apply(ast.literal_eval)

In [28]:
print(roles_df["extracted_skills"].iloc[0])
print(type(roles_df["extracted_skills"].iloc[0]))

['Full-stack engineering', 'Artificial Intelligence (AI)', 'Machine Learning (ML)', 'Large Language Models (LLM)', 'Retrieval-Augmented Generation (RAG)', 'Prompt engineering', 'Retrieval systems', 'Orchestration', 'Data system design', 'Data pipelines', 'Data quality', 'Data lineage', 'Evaluation design', 'Golden datasets', 'Regression evaluation', 'Scenario testing', 'Grounding', 'Hallucination control', 'Guardrails', 'AI safety', 'Model monitoring', 'Drift detection', 'Production monitoring', 'Agentic workflows', 'Agent optimization', 'Non-deterministic systems', 'AI quality management', 'Production AI systems']
<class 'list'>


# berapa persen skill job yang ditemukan di CV

In [22]:
def calculate_skill_match(cv_skills, job_skills):
    cv_skills_lower = [
        skill.lower()
        for skill in cv_skills
    ]

    job_skills_lower = [
        skill.lower()
        for skill in job_skills
    ]

    matched_skills = [
        skill
        for skill in job_skills
        if skill.lower() in cv_skills_lower
    ]

    if len(job_skills_lower) == 0:
        return 0, []
    
    score = len(matched_skills) / len(job_skills_lower)

    return score, matched_skills

In [29]:
#testing

job_skills = roles_df.loc[roles_df["role"] == "Junior Payroll Specialist","extracted_skills"].iloc[0]

score, matched = calculate_skill_match(cv_skills,job_skills)

print("Score: ", score)
print("Matched skills: ", matched)

Score:  0.047619047619047616
Matched skills:  ['Time Management']


In [30]:
#apply ke 9 role

results = []

for _, row in roles_df.iterrows():

    score, matched = calculate_skill_match(
        cv_skills,
        row["extracted_skills"]
    )

    results.append({
        "role": row["role"],
        "skill_score": score,
        "matched_skills": matched
    })

skill_results = pd.DataFrame(results)

skill_results = skill_results.sort_values(
    "skill_score",
    ascending=False
).reset_index(drop=True)

skill_results

,role,skill_score,matched_skills
0,Junior Payroll Specialist,0.047619,[Time Management]
1,Project Manager,0.043478,[Data Analysis]
2,IT Auditor Lead,0.037037,[Data Analysis]
3,AI and Data Specialist,0.000000,[]
4,Technical Support,0.000000,[]
5,SEO Associate,0.000000,[]
6,Solution Engineering,0.000000,[]
7,Risk Management Specialist,0.000000,[]
8,Senior QA Engineer,0.000000,[]


In [31]:
# load embedding model

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [33]:
# melihat perilaku model terhadap skill yang mirip dan tidak mirip.

from sklearn.metrics.pairwise import cosine_similarity

skill_pairs = [
    # kemungkinan MATCH
    ("Data Analysis", "Analytical Skills"),
    ("Customer Service", "Client Relations"),
    ("Payroll Administration", "Payroll Processing"),
    ("Machine Learning", "Artificial Intelligence"),
    ("Selenium", "Automated Testing"),

    # kemungkinan TIDAK MATCH
    ("Python", "Payroll Compliance"),
    ("SEO Writing", "Selenium"),
    ("Machine Learning", "Payroll Processing"),
    ("Accounting", "Playwright"),
    ("Risk Management", "Graphic Design")
]

for skill_a, skill_b in skill_pairs:

    embedding_a = embedding_model.encode(skill_a)
    embedding_b = embedding_model.encode(skill_b)

    similarity = cosine_similarity(
        [embedding_a],
        [embedding_b]
    )[0][0]

    print(f"{skill_a} ↔ {skill_b}: {similarity:.3f}")

Data Analysis ↔ Analytical Skills: 0.330
Customer Service ↔ Client Relations: 0.387
Payroll Administration ↔ Payroll Processing: 0.768
Machine Learning ↔ Artificial Intelligence: 0.703
Selenium ↔ Automated Testing: 0.212
Python ↔ Payroll Compliance: 0.075
SEO Writing ↔ Selenium: 0.177
Machine Learning ↔ Payroll Processing: 0.216
Accounting ↔ Playwright: 0.188
Risk Management ↔ Graphic Design: 0.242


- Selenium ↔ Automated Testing hanya 0.212 . harusnya matchnya tinggi karena untuk automated testing membutuhkan selenium.
- embedding model tidak selalu memahami hubungan skill scr sempurna

# semantic similarity : skill cv vs 176 job skill

In [38]:
roles_df["extracted_skilss"]

0    ['Full-stack engineering', 'AI/ML', 'LLM', 'RA...
1    ['Payroll processing', 'Employment tax law', '...
2    ['Information Systems', 'Logistics Management'...
3    ['SEO campaign optimization', 'SEO content wri...
4    ['JavaScript', 'CSS', 'REST APIs', 'SQL', 'Dat...
5    ['Risk management', 'SaaS', 'Stakeholder manag...
6    ['Project Management', 'ERP Implementation', '...
7    ['IT Audit', 'IT Risk', 'Cybersecurity', 'Tech...
8    ['Selenium', 'Playwright', 'Cypress', 'Appium'...
Name: extracted_skilss, dtype: object

In [40]:
skill_rows = []

for _, row in roles_df.iterrows():
    for skill in row["extracted_skills"]:
        skill_rows.append({
            "role": row["role"],
            "department": row["department"],
            "skill": skill
        })

skills_df = pd.DataFrame(skill_rows)

In [41]:
skills_df.head()

,role,department,skill
0,AI and Data Specialist,Information Technology,Full-stack engineering
1,AI and Data Specialist,Information Technology,Artificial Intelligence (AI)
2,AI and Data Specialist,Information Technology,Machine Learning (ML)
3,AI and Data Specialist,Information Technology,Large Language Models (LLM)
4,AI and Data Specialist,Information Technology,Retrieval-Augmented Generation (RAG)


In [42]:
job_skills_all = skills_df["skill"].tolist()
print(len(job_skills_all))

176


- **cv_skill = isinya dari dataset resume (resume.csv)**
- **job_skill = isinya dataset company roles**

In [43]:
job_skill_embedding = embedding_model.encode(
    job_skills_all,
    show_progress_bar=False
)

print(job_skill_embedding.shape)

(176, 384)


In [44]:
cv_skill_embedding = embedding_model.encode(
    cv_skills,
    show_progress_bar=False
)

print(cv_skill_embedding.shape)

(30, 384)


In [45]:
similarity_matrix = cosine_similarity(
    cv_skill_embedding,
    job_skill_embedding
)

print(similarity_matrix.shape)

(30, 176)


In [46]:
cv_skill_index = 0

scores = similarity_matrix[cv_skill_index]

top_indices = scores.argsort()[-5:][::-1]

for index in top_indices:
    print(
        cv_skills[cv_skill_index],
        "→",
        job_skills_all[index],
        ":",
        round(scores[index], 3)
    )

Customer Service → Information Systems : 0.448
Customer Service → Client Management : 0.426
Customer Service → E-Commerce : 0.418
Customer Service → AWS : 0.404
Customer Service → ERP Consulting : 0.393


In [ ]:
# cek apakah embedding menghasilkan pasangan yang masuk akal.
for i, cv_skill in enumerate(cv_skills):

    scores = similarity_matrix[i]

    top_indices = scores.argsort()[-3:][::-1]

    print(f"\n{cv_skill}")
    
    for index in top_indices:
        print(
            f"  → {job_skills_all[index]} "
            f"({scores[index]:.3f})"
        )


Customer Service
  → Information Systems (0.448)
  → Client Management (0.426)
  → E-Commerce (0.418)

Team Management
  → Teamwork (0.716)
  → Teamwork (0.716)
  → Project Management (0.607)

Marketing & Advertising
  → SEO Campaign Optimization (0.447)
  → E-Commerce (0.435)
  → Communication (0.390)

Conflict Resolution
  → Proactive Issue Resolution (0.535)
  → Coordination (0.531)
  → Coordination (0.531)

Training & Development
  → Organizational Skills (0.494)
  → Technology Assurance (0.452)
  → Learning Agility (0.434)

Client Relations
  → Client Management (0.733)
  → SaaS Domain Knowledge (0.374)
  → ERP Consulting (0.372)

Hospitality Property Management Systems
  → Content Management Systems (0.449)
  → Logistics Management (0.403)
  → Inventory Management (0.397)

Medical Billing & Coding
  → Payroll Software (0.439)
  → Payroll Processing (0.371)
  → Accounting Information Systems (0.342)

Medical Terminology
  → MySQL (0.216)
  → MySQL (0.216)
  → Attention to Detail 

In [48]:
semantic_matches = []

for i, cv_skill in enumerate(cv_skills):

    scores = similarity_matrix[i]

    best_index = scores.argmax()

    semantic_matches.append({
        "cv_skill": cv_skill,
        "job_skill": job_skills_all[best_index],
        "similarity": scores[best_index]
    })

semantic_matches_df = pd.DataFrame(semantic_matches)

semantic_matches_df

,cv_skill,job_skill,similarity
0,Customer Service,Information Systems,0.448419
1,Team Management,Teamwork,0.715756
2,Marketing & Advertising,SEO Campaign Optimization,0.446668
3,Conflict Resolution,Proactive Issue Resolution,0.534792
4,Training & Development,Organizational Skills,0.494424
5,Client Relations,Client Management,0.733088
6,Hospitality Property Management Systems,Content Management Systems,0.448762
7,Medical Billing & Coding,Payroll Software,0.439070
8,Medical Terminology,MySQL,0.216167
9,Data Analysis,Data Analysis,1.000000


# cari semantic match per Role

In [49]:
role_skill_embeddings = {}

for role in skills_df["role"].unique():

    role_skills = skills_df[
        skills_df["role"] == role
    ]["skill"].tolist()

    role_skill_embeddings[role] = (
        role_skills,
        embedding_model.encode(
            role_skills,
            show_progress_bar=False
        )
    )

In [50]:
for role, (skills, embeddings) in role_skill_embeddings.items():
    print(
        role,
        "→",
        len(skills),
        "skills",
        embeddings.shape
    )

AI and Data Specialist → 28 skills (28, 384)
Junior Payroll Specialist → 21 skills (21, 384)
Technical Support → 8 skills (8, 384)
SEO Associate → 16 skills (16, 384)
Solution Engineering → 10 skills (10, 384)
Risk Management Specialist → 18 skills (18, 384)
Project Manager → 23 skills (23, 384)
IT Auditor Lead → 27 skills (27, 384)
Senior QA Engineer → 25 skills (25, 384)


In [52]:
role = "Project Manager"

role_skills, role_embeddings = role_skill_embeddings[role]

similarities = cosine_similarity(
    cv_skill_embedding,
    role_embeddings
)

print(similarities.shape)

(30, 23)


In [53]:
for i, cv_skill in enumerate(cv_skills):

    scores = similarities[i]

    best_index = scores.argmax()

    print(
        cv_skill,
        "→",
        role_skills[best_index],
        ":",
        round(scores[best_index], 3)
    )

Customer Service → Client Management : 0.426
Team Management → Teamwork : 0.716
Marketing & Advertising → Procurement Processes : 0.297
Conflict Resolution → Teamwork : 0.424
Training & Development → Project Management : 0.42
Client Relations → Client Management : 0.733
Hospitality Property Management Systems → Inventory Management : 0.397
Medical Billing & Coding → Microsoft Office : 0.279
Medical Terminology → Odoo : 0.171
Data Analysis → Data Analysis : 1.0
Benefits Administration → SAP : 0.312
Employee Relations → ERP Consulting : 0.363
Labor Relations → Teamwork : 0.33
Policy Development → Project Management : 0.355
Payroll Administration → Microsoft Office : 0.356
Financial Management → Finance Processes : 0.628
Budgeting → Finance Processes : 0.453
Accounting → Finance Processes : 0.459
Purchasing & Procurement → Procurement Processes : 0.808
Public Relations → Communication Skills : 0.413
Government Relations → Project Management : 0.26
Graphic Design → Data Analysis : 0.37
Web

In [54]:
best_scores = []

for i, cv_skill in enumerate(cv_skills):

    scores = similarities[i]

    best_score = scores.max()

    best_scores.append(best_score)

semantic_skill_score = sum(best_scores) / len(best_scores)

print("Semantic Skill Score:", semantic_skill_score)

Semantic Skill Score: 0.44912103911240897


In [56]:
role_semantic_scores = []

for role, (role_skills, role_embeddings) in role_skill_embeddings.items():

    similarities = cosine_similarity(
        cv_skill_embedding,
        role_embeddings
    )

    best_scores = similarities.max(axis=1)

    semantic_score = best_scores.mean()

    role_semantic_scores.append({
        "role": role,
        "semantic_skill_score": semantic_score
    })

role_semantic_df = pd.DataFrame(
    role_semantic_scores
).sort_values(
    "semantic_skill_score",
    ascending=False
).reset_index(drop=True)

role_semantic_df

,role,semantic_skill_score
0,Project Manager,0.449121
1,Junior Payroll Specialist,0.418945
2,IT Auditor Lead,0.415971
3,Solution Engineering,0.386108
4,Risk Management Specialist,0.384739
5,Technical Support,0.382343
6,SEO Associate,0.329639
7,AI and Data Specialist,0.312603
8,Senior QA Engineer,0.274180


In [57]:
import joblib

model = joblib.load("../models/logistic_regression.pkl")
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

In [59]:
import re

def clean_resume(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s+#.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [60]:
sample_cv_clean = clean_resume(sample_cv)

sample_cv_tfidf = tfidf.transform(
    [sample_cv_clean]
)

predicted_category = model.predict(
    sample_cv_tfidf
)[0]

print("Predicted category:", predicted_category)

Predicted category: HR


In [61]:
print(sorted(df["Category"].unique()))

['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


In [62]:
print("Jumlah kategori:", df["Category"].nunique())

Jumlah kategori: 24


In [63]:
for category in sorted(df["Category"].unique()):
    print(category)

ACCOUNTANT
ADVOCATE
AGRICULTURE
APPAREL
ARTS
AUTOMOBILE
AVIATION
BANKING
BPO
BUSINESS-DEVELOPMENT
CHEF
CONSTRUCTION
CONSULTANT
DESIGNER
DIGITAL-MEDIA
ENGINEERING
FINANCE
FITNESS
HEALTHCARE
HR
INFORMATION-TECHNOLOGY
PUBLIC-RELATIONS
SALES
TEACHER


In [64]:
# mapping 24 role di dataset ke 9 role di perusahaan

role_category_mapping = {
    "AI and Data Specialist": [
        "INFORMATION-TECHNOLOGY",
        "ENGINEERING",
        "CONSULTANT"
    ],

    "Junior Payroll Specialist": [
        "HR",
        "FINANCE",
        "ACCOUNTANT"
    ],

    "Technical Support": [
        "INFORMATION-TECHNOLOGY",
        "ENGINEERING",
        "BPO"
    ],

    "SEO Associate": [
        "DIGITAL-MEDIA",
        "DESIGNER",
        "BUSINESS-DEVELOPMENT",
        "PUBLIC-RELATIONS",
        "SALES"
    ],

    "Solution Engineering": [
        "INFORMATION-TECHNOLOGY",
        "ENGINEERING",
        "CONSULTANT"
    ],

    "Risk Management Specialist": [
        "FINANCE",
        "ACCOUNTANT",
        "BANKING",
        "CONSULTANT"
    ],

    "Project Manager": [
        "BUSINESS-DEVELOPMENT",
        "CONSULTANT",
        "ENGINEERING",
        "INFORMATION-TECHNOLOGY",
        "FINANCE"
    ],

    "IT Auditor Lead": [
        "INFORMATION-TECHNOLOGY",
        "ACCOUNTANT",
        "FINANCE",
        "BANKING",
        "CONSULTANT"
    ],

    "Senior QA Engineer": [
        "INFORMATION-TECHNOLOGY",
        "ENGINEERING"
    ]
}

In [65]:
predicted_category = "HR"

for role, categories in role_category_mapping.items():

    if predicted_category in categories:
        print(role, "→ MATCH")
    else:
        print(role, "→ NO MATCH")

AI and Data Specialist → NO MATCH
Junior Payroll Specialist → MATCH
Technical Support → NO MATCH
SEO Associate → NO MATCH
Solution Engineering → NO MATCH
Risk Management Specialist → NO MATCH
Project Manager → NO MATCH
IT Auditor Lead → NO MATCH
Senior QA Engineer → NO MATCH


In [66]:
category_scores = []

for role, categories in role_category_mapping.items():

    if predicted_category in categories:
        score = 1
    else:
        score = 0

    category_scores.append({
        "role": role,
        "category_score": score
    })

category_score_df = pd.DataFrame(category_scores)

category_score_df

,role,category_score
0,AI and Data Specialist,0
1,Junior Payroll Specialist,1
2,Technical Support,0
3,SEO Associate,0
4,Solution Engineering,0
5,Risk Management Specialist,0
6,Project Manager,0
7,IT Auditor Lead,0
8,Senior QA Engineer,0


In [67]:
hybrid_df = role_semantic_df.merge(
    category_score_df,
    on="role"
)

hybrid_df

,role,semantic_skill_score,category_score
0,Project Manager,0.449121,0
1,Junior Payroll Specialist,0.418945,1
2,IT Auditor Lead,0.415971,0
3,Solution Engineering,0.386108,0
4,Risk Management Specialist,0.384739,0
5,Technical Support,0.382343,0
6,SEO Associate,0.329639,0
7,AI and Data Specialist,0.312603,0
8,Senior QA Engineer,0.274180,0


In [68]:
hybrid_df["hybrid_score"] = (
    0.8 * hybrid_df["semantic_skill_score"]
    + 0.2 * hybrid_df["category_score"]
)

hybrid_df = hybrid_df.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(drop=True)

hybrid_df

,role,semantic_skill_score,category_score,hybrid_score
0,Junior Payroll Specialist,0.418945,1,0.535156
1,Project Manager,0.449121,0,0.359297
2,IT Auditor Lead,0.415971,0,0.332777
3,Solution Engineering,0.386108,0,0.308886
4,Risk Management Specialist,0.384739,0,0.307791
5,Technical Support,0.382343,0,0.305874
6,SEO Associate,0.329639,0,0.263711
7,AI and Data Specialist,0.312603,0,0.250082
8,Senior QA Engineer,0.274180,0,0.219344


In [86]:
def recommend_roles_v2(cv_text):

    # ==========================================
    # 1. EXTRACT SKILLS DARI CV
    # ==========================================

    cv_skills = extract_cv_skills(cv_text)

    # ==========================================
    # 2. SEMANTIC MATCHING
    # ==========================================

    cv_skill_embeddings = embedding_model.encode(
        cv_skills,
        show_progress_bar=False
    )

    role_scores = []

    for role, (role_skills, role_embeddings) in role_skill_embeddings.items():

        similarities = cosine_similarity(
            cv_skill_embeddings,
            role_embeddings
        )

        # Untuk setiap skill CV,
        # ambil skill job yang paling mirip
        best_scores = similarities.max(axis=1)

        # Rata-rata seluruh skill CV
        semantic_score = best_scores.mean()

        role_scores.append({
            "role": role,
            "semantic_skill_score": semantic_score
        })

    result = pd.DataFrame(role_scores)

    # ==========================================
    # 3. PREDIKSI KATEGORI DENGAN ML
    # ==========================================

    cv_clean = clean_resume(cv_text)

    cv_tfidf = tfidf.transform(
        [cv_clean]
    )

    predicted_category = model.predict(
        cv_tfidf
    )[0]

    # ==========================================
    # 4. CATEGORY SCORE
    # ==========================================

    category_scores = []

    for role, categories in role_category_mapping.items():

        if predicted_category in categories:
            score = 1
        else:
            score = 0

        category_scores.append({
            "role": role,
            "category_score": score
        })

    category_score_df = pd.DataFrame(
        category_scores
    )

    # ==========================================
    # 5. GABUNGKAN SEMANTIC + CATEGORY
    # ==========================================

    result = result.merge(
        category_score_df,
        on="role"
    )

    # ==========================================
    # 6. HYBRID SCORE
    # ==========================================

    result["hybrid_score"] = (
        0.8 * result["semantic_skill_score"]
        + 0.2 * result["category_score"]
    )

    # ==========================================
    # 7. SORTING
    # ==========================================

    result = result.sort_values(
        "hybrid_score",
        ascending=False
    ).reset_index(drop=True)

    return predicted_category, result

In [87]:
predicted_category, result = recommend_roles_v2(sample_cv)

print("Predicted category:", predicted_category)

result

Predicted category: HR


,role,semantic_skill_score,category_score,hybrid_score
0,Junior Payroll Specialist,0.451624,1,0.561299
1,Project Manager,0.442674,0,0.354139
2,IT Auditor Lead,0.427017,0,0.341614
3,Risk Management Specialist,0.396113,0,0.316891
4,Technical Support,0.392154,0,0.313723
5,Solution Engineering,0.379420,0,0.303536
6,SEO Associate,0.351312,0,0.281049
7,AI and Data Specialist,0.309854,0,0.247883
8,Senior QA Engineer,0.291973,0,0.233578


In [91]:
test_results = []
failed_cvs = []

for i in range(10):

    print(f"\nProcessing CV {i}...")

    cv_text = df.iloc[i]["Resume_str"]
    actual_category = df.iloc[i]["Category"]

    try:

        predicted_category, result = recommend_roles_v2(cv_text)

        top_role = result.iloc[0]["role"]
        top_score = result.iloc[0]["hybrid_score"]

        test_results.append({
            "index": i,
            "actual_category": actual_category,
            "predicted_category": predicted_category,
            "top_role": top_role,
            "hybrid_score": top_score
        })

        print("Berhasil")
        print("Predicted:", predicted_category)
        print("Top role:", top_role)

    except Exception as e:

        print("Gagal")
        print("Error:", e)

        failed_cvs.append(i)


Processing CV 0...
Response tidak memiliki 'choices':
{'id': 'gen-1791285017-e7cjAYGT6vvGtaTMptpv', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}
Gagal
Error: 'NoneType' object is not subscriptable

Processing CV 1...
Berhasil
Predicted: PUBLIC-RELATIONS
Top role: SEO Associate

Processing CV 2...
Berhasil
Predicted: HR
Top role: Junior Payroll Specialist

Processing CV 3...
Response tidak memiliki 'choices':
{'id': 'gen-1791285389-6sk3AkJJDh2rlSQOPtxQ', 'error': {'message': 'Upstream error from Nvidia: Service temporarily overloaded', 'code': 503, 'metadata': {'error_type': 'provider_overloaded'}}}
Gagal
Error: 'NoneType' object is not subscriptable

Processing CV 4...
Berhasil
Predicted: HR
Top role: Junior Payroll Specialist

Processing CV 5...
Status code: 429
Response: {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests

In [92]:
print("CV berhasil:", len(test_results))
print("CV gagal:", failed_cvs)

CV berhasil: 3
CV gagal: [0, 3, 5, 6, 7, 8, 9]


In [95]:
import time

retry_results = []

for i in failed_cvs:

    print(f"\nRetry CV {i}...")

    cv_text = df.iloc[i]["Resume_str"]
    actual_category = df.iloc[i]["Category"]

    try:

        predicted_category, result = recommend_roles_v2(cv_text)

        top_role = result.iloc[0]["role"]
        top_score = result.iloc[0]["hybrid_score"]

        retry_results.append({
            "index": i,
            "actual_category": actual_category,
            "predicted_category": predicted_category,
            "top_role": top_role,
            "hybrid_score": top_score
        })

        print("Berhasil")

    except Exception as e:

        print("Masih gagal")
        print("Error:", e)

    time.sleep(10)


Retry CV 0...
Status code: 429
Response: {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1791331200000"},"limit_source":"openrouter_free_tier_daily","remedy_hint":"Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.","provider_name":null}},"user_id":"user_3JOyzkKB9JZzHkHDh1YziPH42VL"}
Masih gagal
Error: 'NoneType' object is not subscriptable

Retry CV 3...
Status code: 429
Response: {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1791331200000"},"limit_source":"openrouter_free_tier_daily","remedy_hint":"Wait for the daily reset (see X-RateLimit-Reset), or purchase credits 